# Latencia de extremo a extremo — Objetivo específico 4 / Hito H4

Mide el pipeline completo, de archivo de audio a archivo BRF, para verificar el criterio
del cuarto objetivo específico.

> El sistema debe entregar un BRF sintácticamente válido en un tiempo **menor o igual a
> 1.5 veces la duración del audio**, en al menos el **90 % de las ejecuciones**, para
> fragmentos de hasta cinco minutos.

Se cronometra cada etapa por separado, de modo que el resultado diga no solo si se cumple
sino dónde se consume el tiempo. La etapa determinista ya se midió en local y resulta
despreciable, del orden de 15 ms para cinco minutos de audio; esta corrida lo confirma
sobre audio real y añade la etapa acústica, que es la que domina.

El código se toma del repositorio para que la medición corresponda exactamente al sistema
entregado, sin reimplementaciones.

## 1. Repositorio y dependencias

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = "https://github.com/G2309/amt-braille-translator.git"
BRANCH = "dev"
WORK = Path("/kaggle/working/amt-braille-translator")

if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORK)], check=True)
sys.path.insert(0, str(WORK / "src"))

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("--no-deps", "piano_transcription_inference", "torchlibrosa")
pip("librosa", "soundfile")
print("commit:", subprocess.run(["git", "-C", str(WORK), "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

## 2. Conjunto de prueba\n\nEl criterio aplica a fragmentos de hasta cinco minutos, así que se recortan segmentos de duración creciente sobre varias obras distintas. Así se comprueba además si el cociente se degrada con la duración.

In [ ]:
import csv, glob, random
import librosa, soundfile as sf

DURACIONES = [30, 60, 120, 180, 300]   # segundos, dentro del alcance declarado
N_OBRAS = 4                            # obras distintas por duracion -> 20 ejecuciones
SEED = 22779

DATASET_ROOT = None
for cand in glob.glob("/kaggle/input/*/"):
    hits = glob.glob(cand + "**/maestro-v3.0.0.csv", recursive=True)
    if hits:
        DATASET_ROOT = Path(hits[0]).parent
        break
assert DATASET_ROOT is not None, "Montar el dataset the-maestro-dataset-v3-0-0 como input"

with open(DATASET_ROOT / "maestro-v3.0.0.csv", newline="", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if r["split"] == "test"]

# solo obras suficientemente largas para recortar el segmento mayor
largas = [r for r in rows if float(r["duration"]) >= max(DURACIONES) + 5]
obras = random.Random(SEED).sample(largas, N_OBRAS)

def resolve(rel):
    p = DATASET_ROOT / rel
    if p.exists():
        return p
    hits = glob.glob(str(DATASET_ROOT / "**" / Path(rel).name), recursive=True)
    assert hits, f"No se encontro {rel}"
    return Path(hits[0])

SEGMENTOS = Path("/kaggle/working/segmentos")
SEGMENTOS.mkdir(exist_ok=True)

casos = []
for obra in obras:
    origen = resolve(obra["audio_filename"])
    for dur in DURACIONES:
        destino = SEGMENTOS / f"{Path(obra['midi_filename']).stem}__{dur}s.wav"
        if not destino.exists():
            audio, sr = librosa.load(str(origen), sr=None, mono=True, duration=dur)
            sf.write(str(destino), audio, sr)
        casos.append({"path": destino, "duracion_objetivo": dur,
                      "compositor": obra["canonical_composer"],
                      "obra": obra["canonical_title"]})

print(f"{len(casos)} segmentos preparados sobre {N_OBRAS} obras")

## 3. Medición\n\nEl modelo se carga una sola vez antes de cronometrar, para que el tiempo de carga del checkpoint no se cuente como latencia de proceso.

In [ ]:
import torch
from amt.transcriber import AMTTranscriber
from evaluation.latency import LatencySummary, measure_end_to_end

device = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", device)

transcriptor = AMTTranscriber(device=device)
# calentamiento: descarga y carga del checkpoint fuera de la medicion
transcriptor.transcribe(str(casos[0]["path"]))
print("modelo listo")

In [ ]:
SALIDAS = Path("/kaggle/working/brf")
SALIDAS.mkdir(exist_ok=True)

runs, filas = [], []
for caso in casos:
    destino_brf = SALIDAS / (caso["path"].stem + ".brf")
    run = measure_end_to_end(str(caso["path"]), transcriptor.transcribe,
                             output_path=str(destino_brf))
    runs.append(run)
    fila = run.as_dict()
    fila.update({"duracion_objetivo": caso["duracion_objetivo"],
                 "compositor": caso["compositor"], "obra": caso["obra"],
                 "brf": str(destino_brf)})
    filas.append(fila)
    print(f"{caso['duracion_objetivo']:>4}s  {caso['compositor'][:18]:18s} "
          f"amt={run.timings.amt_s:6.2f}s  det={run.timings.deterministic_s:6.3f}s  "
          f"total={run.timings.total_s:6.2f}s  x={run.ratio:.4f}  "
          f"{'ok' if run.within_threshold else 'EXCEDE'}")

## 4. Verificación del criterio

In [ ]:
resumen = LatencySummary(runs)
print(resumen.report())

## 5. Validez sintáctica de los BRF generados\n\nEl objetivo exige que el archivo entregado sea sintácticamente válido, no solo que llegue a tiempo.

In [ ]:
from braille_translator.brf_exporter import validate_brf

invalidos = 0
for fila in filas:
    problemas = validate_brf(fila["brf"])
    fila["brf_valido"] = not problemas
    fila["brf_problemas"] = "; ".join(problemas)
    if problemas:
        invalidos += 1
        print(f"{Path(fila['brf']).name}: {problemas[:3]}")

print(f"\nBRF validos: {len(filas) - invalidos}/{len(filas)}")

## 6. Resultados por duración y exportación

In [ ]:
import pandas as pd

df = pd.DataFrame(filas)
por_duracion = df.groupby("duracion_objetivo").agg(
    ejecuciones=("ratio", "count"),
    amt_s=("amt_s", "mean"),
    deterministic_s=("deterministic_s", "mean"),
    total_s=("total_s", "mean"),
    ratio_medio=("ratio", "mean"),
    ratio_max=("ratio", "max"),
).round(4)
display(por_duracion)

df.to_csv("/kaggle/working/latencia_e2e.csv", index=False)
por_duracion.to_csv("/kaggle/working/latencia_por_duracion.csv")
print("CSV escritos")

## 7. Trazabilidad

- El criterio verificado corresponde al cuarto objetivo específico y al hito H4.
- La medición usa el código del repositorio en el commit impreso en la celda 1, de modo
  que es reproducible y corresponde al sistema entregado.
- `latencia_e2e.csv` guarda una fila por ejecución con el desglose por etapa; con eso se
  redacta la sección de resultados sin volver a ejecutar nada.